In [38]:
import os
import json
import ollama

from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents

In [39]:
import sys
!{sys.executable} -m pip install python-dotenv

In [40]:
MODEL = "llama3.2"

print("Ollama is ready!")

Ollama is ready!


In [41]:
links = fetch_website_links("https://huggingface.co")
links

['/',
 '/models',
 '/datasets',
 '/spaces',
 '/storage',
 '/docs',
 '/enterprise',
 '/pricing',
 '/tasks',
 '/chat',
 '/collections',
 '/languages',
 '/organizations',
 '/blog',
 '/posts',
 '/papers',
 '/hardware',
 '/learn',
 '/join/discord',
 'https://discuss.huggingface.co/',
 'https://github.com/huggingface',
 '/enterprise',
 '/pro',
 '/support',
 '/inference/models',
 '/inference-endpoints',
 '/storage',
 '/login',
 '/join',
 'https://pollen-robotics.com/microduck',
 '/spaces',
 '/models',
 '/zai-org/GLM-5.3',
 '/Qwen/Qwen3.8-Flash-Next',
 '/zai-org/GLM-5.3-Flash',
 '/Qwen/Qwen3.8-27B',
 '/deepseek-ai/DeepSeek-V4-Flash-Vision-Exp',
 '/models',
 '/spaces/pollen-robotics/microduck-simulator',
 '/spaces/kulkas2pintu/wan555',
 '/spaces/kulkas2pintu/QWEN_EDIT_IMAGE',
 '/spaces/Saravutw/Omni-videos-custom',
 '/spaces/MiniMaxAI/MiniMax-H3-Turbo-Lora',
 '/spaces',
 '/datasets/markov-ai/cad-1000-hours',
 '/datasets/rajpurkar/squad',
 '/datasets/stanfordnlp/imdb',
 '/datasets/nyu-mll/glue',

In [42]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [43]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [44]:
print(get_links_user_prompt("https://huggingface.co"))


Here is the list of links on the website https://huggingface.co -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

/
/models
/datasets
/spaces
/storage
/docs
/enterprise
/pricing
/tasks
/chat
/collections
/languages
/organizations
/blog
/posts
/papers
/hardware
/learn
/join/discord
https://discuss.huggingface.co/
https://github.com/huggingface
/enterprise
/pro
/support
/inference/models
/inference-endpoints
/storage
/login
/join
https://pollen-robotics.com/microduck
/spaces
/models
/zai-org/GLM-5.3
/Qwen/Qwen3.8-Flash-Next
/zai-org/GLM-5.3-Flash
/Qwen/Qwen3.8-27B
/deepseek-ai/DeepSeek-V4-Flash-Vision-Exp
/models
/spaces/pollen-robotics/microduck-simulator
/spaces/kulkas2pintu/wan555
/spaces/kulkas2pintu/QWEN_EDIT_IMAGE
/spaces/Saravutw/Omni-videos-custom
/spaces/MiniMaxAI/MiniMax-H3-Turbo-Lora
/spaces
/data

In [45]:
import re
from urllib.parse import urljoin

def select_relevant_links(url):

    response = ollama.chat(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": link_system_prompt
            },
            {
                "role": "user",
                "content": get_links_user_prompt(url)
            }
        ],
        format="json"
    )

    result = response["message"]["content"]
    links = json.loads(result)

    cleaned_links = []

    for link in links.get("links", []):

        if not isinstance(link, dict):
            continue

        link_url = link.get("url", "").strip()

        match = re.search(r'\]\((https?://[^)]+)\)', link_url)

        if match:
            link_url = match.group(1)

        elif link_url.startswith("/"):
            link_url = urljoin(url, link_url)

        if (
            link_url.startswith(("http://", "https://"))
            and "terms" not in link_url.lower()
            and "privacy" not in link_url.lower()
        ):
            cleaned_links.append({
                "type": link.get("type", "other"),
                "url": link_url
            })

    return {"links": cleaned_links}

In [46]:
links = select_relevant_links("https://huggingface.co")
print(links)

{'links': [{'type': 'Company page', 'url': 'https://huggingface.co/'}, {'type': 'Blog page', 'url': 'https://blog.huggingface.co/'}, {'type': 'About page', 'url': 'https://huggingface.co/team'}, {'type': 'About page', 'url': 'https://huggingface.co/mission'}, {'type': 'FAQ/Support page', 'url': 'https://support.huggingface.co/'}, {'type': 'Careers/Jobs page', 'url': 'https://apply.workable.com/huggingface'}, {'type': 'GitHub page', 'url': 'https://github.com/huggingface'}, {'type': 'Discord page', 'url': 'https://join.discord.com/huggingface'}, {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'}, {'type': 'LinkedIn page', 'url': 'https://www.linkedin.com/company/huggingface/'}, {'type': 'Changelog page', 'url': 'https://changelog.huggingface.co/'}, {'type': 'Documentation page', 'url': 'https://docs.huggingface.co/'}, {'type': 'Pricing page', 'url': 'https://huggingface.co/pricing'}, {'type': 'Models page', 'url': 'https://huggingface.co/models'}, {'type': 'Datasets page

In [47]:
def fetch_page_and_all_relevant_links(url):

    contents = fetch_website_contents(url)

    relevant_links = select_relevant_links(url)

    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"

    for link in relevant_links["links"]:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])

    return result

In [48]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Skipping inaccessible link: https://docs.huggingface.co
Reason: HTTPSConnectionPool(host='docs.huggingface.co', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("HTTPSConnection(host='docs.huggingface.co', port=443): Failed to resolve 'docs.huggingface.co' ([Errno 11001] getaddrinfo failed)"))
Skipping inaccessible link: https://blog.huggingface.co
Reason: HTTPSConnectionPool(host='blog.huggingface.co', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("HTTPSConnection(host='blog.huggingface.co', port=443): Failed to resolve 'blog.huggingface.co' ([Errno 11001] getaddrinfo failed)"))
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Provider

In [49]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [50]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [51]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Skipping inaccessible link: https://blog.huggingface.co/
Reason: HTTPSConnectionPool(host='blog.huggingface.co', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("HTTPSConnection(host='blog.huggingface.co', port=443): Failed to resolve 'blog.huggingface.co' ([Errno 11001] getaddrinfo failed)"))
Skipping inaccessible link: https://join.discord.com/huggingface
Reason: HTTPSConnectionPool(host='join.discord.com', port=443): Max retries exceeded with url: /huggingface (Caused by NameResolutionError("HTTPSConnection(host='join.discord.com', port=443): Failed to resolve 'join.discord.com' ([Errno 11001] getaddrinfo failed)"))


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nNEW\nMicroduck: A Tiny Robot for AI Builders 🦆\nGoogle Gemma 4 is here 💫\nStorage Buckets: AI-native object storage\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\

In [52]:
def create_brochure(company_name, url): 
    response = ollama.chat( 
        model=MODEL, 
        messages=[ 
            { 
                "role": "system", 
                "content": brochure_system_prompt 
            }, 
            { 
                "role": "user", 
                "content": get_brochure_user_prompt(company_name, url) 
            } 
        ] 
    ) 
 
    result = response["message"]["content"] 
    display(Markdown(result))

In [53]:
create_brochure("Hugging Face", "https://huggingface.co")

Skipping inaccessible link: https://blog.huggingface.co/
Reason: HTTPSConnectionPool(host='blog.huggingface.co', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("HTTPSConnection(host='blog.huggingface.co', port=443): Failed to resolve 'blog.huggingface.co' ([Errno 11001] getaddrinfo failed)"))
Skipping inaccessible link: https://docs.huggingface.co/
Reason: HTTPSConnectionPool(host='docs.huggingface.co', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("HTTPSConnection(host='docs.huggingface.co', port=443): Failed to resolve 'docs.huggingface.co' ([Errno 11001] getaddrinfo failed)"))
Skipping inaccessible link: https://join.discord.com/huggingface
Reason: HTTPSConnectionPool(host='join.discord.com', port=443): Max retries exceeded with url: /huggingface (Caused by NameResolutionError("HTTPSConnection(host='join.discord.com', port=443): Failed to resolve 'join.discord.com' ([Errno 11001] getaddrinfo failed)"))
Skipping inaccessible li

# Hugging Face: Building the Future of Artificial Intelligence

[Image: Hugging Face logo]

Welcome to Hugging Face, the leading collaboration platform for the machine learning community. Our mission is to empower researchers, developers, and entrepreneurs to create, discover, and collaborate on AI models, datasets, and applications.

## About Us

Hugging Face is a community-driven platform that provides a comprehensive suite of tools and resources for machine learning. Our platform is designed to facilitate collaboration, innovation, and progress in the field of AI. With a strong focus on community engagement, we strive to create a collaborative environment that fosters knowledge sharing, idea generation, and mutual support.

## Our Mission

Our mission is to build the future of artificial intelligence by providing a platform that enables the machine learning community to come together, share ideas, and create innovative solutions. We aim to make AI more accessible, affordable, and user-friendly, while promoting collaboration, transparency, and accountability.

## Our Products and Services

* **Hugging Face Hub**: Our flagship platform provides a suite of tools and resources for machine learning, including models, datasets, and applications.
* **Models**: Browse 2M+ pre-trained models and fine-tune them for your specific use case.
* **Datasets**: Explore 500k+ datasets and collaborate with others to create and share your own datasets.
* **Spaces**: Host and collaborate on ML projects with others, including text-to-image generation and video generation.
* **Storage Buckets**: AI-native object storage for your models and datasets.
* **Inference Providers**: Get started with our inference providers for easy model deployment and integration.

## Our Community

Our community is at the heart of what we do. We believe that collaboration, transparency, and accountability are essential for building trust and driving progress in the field of AI. Our community includes researchers, developers, entrepreneurs, and enthusiasts who share our passion for AI and machine learning.

## Join Our Community

Join our community today and become part of a vibrant ecosystem that is shaping the future of AI. Collaborate with others, share your ideas, and contribute to the growth of the machine learning community.

## Careers and Opportunities

Are you passionate about AI and machine learning? Do you want to join a dynamic and innovative team that is shaping the future of AI? Check out our job openings and career opportunities.

## Contact Us

Get in touch with us to learn more about Hugging Face and how we can help you achieve your AI goals.

In [54]:
def stream_brochure(company_name, url):

    response = ""

    display_handle = display(Markdown(""), display_id=True)

    stream = ollama.chat(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": brochure_system_prompt
            },
            {
                "role": "user",
                "content": get_brochure_user_prompt(
                    company_name,
                    url
                )
            }
        ],
        stream=True
    )

    for chunk in stream:

        text = chunk["message"]["content"]

        response += text

        update_display(
            Markdown(response),
            display_id=display_handle.display_id
        )

In [55]:
stream_brochure("HuggingFace", "https://huggingface.co")

Hugging Face Brochure
======================

Welcome to Hugging Face, the AI community building the future. We are a collaboration platform where machine learning enthusiasts, researchers, and developers come together to create, discover, and push the boundaries of artificial intelligence.

**Our Mission**

Hugging Face is dedicated to empowering the machine learning community to build, share, and apply AI models, datasets, and applications. We strive to provide a platform that fosters collaboration, innovation, and growth.

**Our Focus**

Our focus areas include:

*   **Models**: We offer a vast library of pre-trained models, datasets, and tools that enable developers to build, train, and deploy AI models.
*   **Datasets**: We provide access to a vast repository of high-quality datasets, which are essential for training and validating AI models.
*   **Spaces**: Our collaboration platform allows users to create, share, and join spaces for discussion, project management, and community building.
*   **Buckets**: We offer AI-native object storage solutions that enable users to store, manage, and deploy AI models and datasets efficiently.

**Community**

Hugging Face is built on the principles of community, collaboration, and mutual support. Our community includes:

*   **Researchers**: We welcome researchers and academics who are working on cutting-edge AI projects and share their knowledge, expertise, and resources with the community.
*   **Developers**: Our community includes developers, engineers, and software professionals who build, deploy, and maintain AI models and applications.
*   **Entrepreneurs**: We support entrepreneurs and startups that are building innovative AI-powered products and services.

**Careers and Jobs**

Join our team of passionate and talented individuals who are shaping the future of AI. We offer a range of career opportunities in:

*   **Research and Development**: We are seeking talented researchers and engineers to work on cutting-edge AI projects.
*   **Software Development**: Our team is looking for skilled software developers to build, maintain, and deploy AI models and applications.
*   **Customer Support**: We need customer support specialists to help our community members with their queries and issues.

**Stay in Touch**

Want to stay up-to-date with the latest news, updates, and releases from Hugging Face? Follow us on social media, subscribe to our newsletter, or visit our blog to learn more about our community, products, and services.

[Learn More](link to learn more)
[Join Our Community](link to join our community)
[Careers](link to careers)

---

We hope you've enjoyed this brief introduction to Hugging Face. We invite you to join our community, explore our products and services, and contribute to the future of AI.

Skipping inaccessible link: https://join.huggingface.co/
Reason: HTTPSConnectionPool(host='join.huggingface.co', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("HTTPSConnection(host='join.huggingface.co', port=443): Failed to resolve 'join.huggingface.co' ([Errno 11001] getaddrinfo failed)"))
